# MachineGuard AI - Exploratory Data Analysis
Run from the `notebooks/` folder or repo root. Outputs are intentionally empty: execute the cells to generate real results.

In [ ]:
import sys; from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name=='notebooks' else Path.cwd()))
import pandas as pd, plotly.express as px
from src.data_loader import load_raw, validate_raw, clean_dataset
from src.config import NUM_FEATURES, TARGET

## 1. Load and validate

In [ ]:
raw = load_raw()
print(validate_raw(raw))
raw.head()

## 2. Structure, types, missing values, duplicates

In [ ]:
print(raw.shape); print(raw.dtypes); print('missing:', raw.isna().sum().sum(), '| duplicates:', raw.duplicated().sum())

## 3. Data leakage check
If the target equals the OR of the five failure-mode flags, those flags encode the target and must not be features.

In [ ]:
modes = ['TWF','HDF','PWF','OSF','RNF']
print((raw[modes].max(axis=1) == raw['Machine failure']).mean())
clean = clean_dataset(raw); print(list(clean.columns))

## 4. Class imbalance

In [ ]:
vc = clean[TARGET].value_counts(); print(vc); print('failure %:', 100*vc[1]/vc.sum())

## 5. Distributions, outliers and failure relationships

In [ ]:
clean[NUM_FEATURES].describe()

In [ ]:
for col in NUM_FEATURES:
    px.histogram(clean, x=col, color=clean[TARGET].astype(str), barmode='overlay', nbins=40, title=col).show()

## 6. Correlation and machine type

In [ ]:
px.imshow(clean[NUM_FEATURES+[TARGET]].corr().round(2), text_auto=True, color_continuous_scale='RdBu_r').show()
clean.groupby('type')[TARGET].agg(['size','mean'])

Next step: `python src/train_models.py` (training, evaluation, tuning, clustering).